In [1]:
# ── Required imports ──
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset, Subset
from torchvision import datasets, transforms, models
from sklearn.model_selection import train_test_split
import pywt

# ── Paste the full data loading block from Part 1 ──
# (everything from img_size = 96 down to the DataLoader definitions)

# ── Define wavelet_compress (copy from the MobileNet section) ──
def wavelet_compress(img_tensor, threshold=0.1, wavelet='db1', level=2):
    img_np = img_tensor.numpy()
    compressed = np.zeros_like(img_np)
    for c in range(3):
        channel = img_np[c]
        coeffs = pywt.wavedec2(channel, wavelet=wavelet, level=level)
        coeffs_thresh = [coeffs[0]]
        for detail in coeffs[1:]:
            coeffs_thresh.append(
                tuple(pywt.threshold(d, value=threshold, mode='hard') for d in detail)
            )
        recon = pywt.waverec2(coeffs_thresh, wavelet=wavelet)
        recon = recon[:channel.shape[0], :channel.shape[1]]
        compressed[c] = recon
    return torch.tensor(compressed, dtype=torch.float32)

In [2]:
# --- CELL 1: FINE-TUNING LOOP ---
from torch.utils.data import DataLoader, Dataset
from torchvision import models
from torch import nn, optim

# 1. The Bridge to the wavelet function
class CompressedDataset(Dataset):
    def __init__(self, original_subset, threshold=0.1):
        self.original_subset = original_subset
        self.threshold = threshold
    def __len__(self): 
        return len(self.original_subset)
    def __getitem__(self, idx):
        img, label = self.original_subset[idx]
        compressed_img = wavelet_compress(img, threshold=self.threshold) 
        return compressed_img, label

# 2. Initialize Data 
loss_history = [] # This is needed for the plots cell later
ds_train_comp = CompressedDataset(set_train, threshold=0.1)
ldr_train_comp = DataLoader(ds_train_comp, batch_size=32, shuffle=True, num_workers=2)

# 3. Setup the Model (ResNet-18)
model_ft = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
model_ft.fc = nn.Linear(model_ft.fc.in_features, n_classes) 
model_ft = model_ft.to(run_device)

# 4. Training Settings
optimizer_ft = optim.AdamW(model_ft.parameters(), lr=1e-4)
criterion = nn.CrossEntropyLoss()

# 5. Training Loop
print("Fine-tuning on Compressed Data...")
for epoch in range(1, 4):
    model_ft.train()
    epoch_loss = 0.0
    for images, labels in ldr_train_comp:
        images, labels = images.to(run_device), labels.to(run_device)
        optimizer_ft.zero_grad()
        loss = criterion(model_ft(images), labels)
        loss.backward()
        optimizer_ft.step()
        
        loss_history.append(loss.item()) # Record for the plot cell
        epoch_loss += loss.item()
    print(f"Epoch {epoch} complete. Avg Loss: {epoch_loss/len(ldr_train_comp):.4f}")

NameError: name 'set_train' is not defined

In [ ]:
# --- CELL 2: MEASUREMENTS (ACCURACY) ---
model_ft.eval()
correct = 0
total = 0

# Prepare the test loader
ds_test_comp = CompressedDataset(set_test, threshold=0.1)
ldr_test_comp = DataLoader(ds_test_comp, batch_size=batch_size, shuffle=False)

print("Calculating accuracy on compressed test set...")
with torch.no_grad():
    for images, labels in ldr_test_comp:
        images, labels = images.to(run_device), labels.to(run_device)
        outputs = model_ft(images)
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

print(f'\nFinal Accuracy of Model 1 on Compressed Images: {100 * correct / total:.2f}%')

In [ ]:
# --- CELL 3: PLOTS ---
import matplotlib.pyplot as plt

plt.figure(figsize=(12, 5))

# 1. Training Loss Plot
plt.subplot(1, 2, 1)
plt.plot(loss_history, color='tab:red')
plt.title("Fine-tuning Loss Curve")
plt.xlabel("Iteration")
plt.ylabel("Loss")

# 2. Visual Comparison Plot
original_img, _ = set_train[0]
compressed_img = wavelet_compress(original_img, threshold=0.1)

plt.subplot(1, 2, 2)
plt.imshow(compressed_img.permute(1, 2, 0).cpu().numpy())
plt.title("Sample: What the Model sees (Compressed)")

plt.tight_layout()
plt.show()